In [3]:
# Adversarial validation
import pandas as pd 
import numpy as np 
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import  make_pipeline
from sklearn.model_selection import  StratifiedKFold, cross_val_score
from feature_eng import pipe_random_tree_new_eng
dev = pd.read_csv("development(1).csv")
# cats - sex, medical_center, dzgroup, dzclass, race, ca, dnr
ev = pd.read_csv("evaluation.csv")

In [4]:
dev_c = dev.copy()
ev_c = ev.copy()
dev_c["is_test"] = 0
ev_c["is_test"] = 1
concat = pd.concat([dev_c.drop(columns=["death"]), ev_c], axis=0, ignore_index=True)
y_adv = concat["is_test"]
x_adv = concat.drop(columns=["is_test"])

mod = make_pipeline(pipe_random_tree_new_eng, RandomForestClassifier(class_weight="balanced", random_state=17, n_estimators=400, n_jobs=-1))
mod.fit(x_adv, y_adv)   
importances = mod.steps[1][1].feature_importances_# must be done here
col_transformer = mod.steps[0][1].steps[2][1] # will allow us to extract the name of the most important feature

scores = cross_val_score(make_pipeline(pipe_random_tree_new_eng, RandomForestClassifier(random_state=17, class_weight="balanced", n_estimators=400, n_jobs=-1)), x_adv, y_adv, cv=StratifiedKFold(10, shuffle=True, random_state=17), scoring="roc_auc")
print(scores.mean()) # cross-validated ROC-AUC score

# Get the feature name from the preprocessor step
feature_names = col_transformer.get_feature_names_out()
top_idx = np.argmax(importances)
top_feature = feature_names[top_idx] # feature that would need to be removed if ROC_AUC >> 0.5

0.5078770429155245
